# Green Gate — 60-second commercial (Blender, GPU render)

This notebook builds the whole commercial in **Blender 4.5 LTS** and renders it on Google's free GPU.
Nothing runs on your laptop.

**How to run**

1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**.
3. When asked, allow Google Drive access (frames are saved there so the render can resume). The project files
   download automatically from GitHub (if that fails, you are asked to choose **`greengate_bundle.zip`**).
4. Leave the tab open. When it finishes, `GreenGate_Commercial.mp4` downloads automatically and is also saved in
   **My Drive → GreenGate_Commercial**.

**How long it takes:** the default *standard* quality needs roughly 5–6 hours of GPU time. Free Colab sessions
usually give a few hours, so expect to run it in two sittings. After the first few frames the notebook prints the
real speed and the time left. Set `QUALITY = "fast"` (about 2.5 h) if you want it in one go.

**If Colab disconnects or says you hit the GPU limit:** open the notebook later and press **Run all** again.
It continues from the last finished frame. Nothing is lost (frames are stored in your Google Drive).

In [ ]:
# ---- Settings -------------------------------------------------------------
QUALITY = "standard"   # "standard" = 1080p, 24 samples  (about 5-6 h on a free T4, resumes across sessions)
                       # "final"    = 1080p, 48 samples  (cleanest glass, about 11 h)
                       # "fast"     = 900p, 16 samples   (about 2.5 h)
                       # "draft"    = 540p, 8 samples    (about 30 min, to check the whole edit)
USE_DRIVE = True       # keep frames in Google Drive so an interrupted render can resume
MAKE_SILENT_VERSION = True   # also export a version without music (to add your own track)

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu or 'No GPU found')
assert 'GPU' in gpu, 'No GPU: use Runtime > Change runtime type > T4 GPU, then Run all again.'

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/GreenGate_Commercial'
else:
    WORK = '/content/GreenGate_Commercial'
os.makedirs(WORK, exist_ok=True)
ZIP = os.path.join(WORK, 'greengate_bundle.zip')
BUNDLE_URL = 'https://github.com/bavumile873-cyber/greengate-commercial/raw/main/greengate_bundle.zip'
if not os.path.exists(ZIP):
    import urllib.request
    try:
        urllib.request.urlretrieve(BUNDLE_URL, ZIP)
        print('Downloaded bundle from GitHub')
    except Exception as e:
        print('GitHub download failed (%s) - upload it instead.' % e)
        from google.colab import files
        print('Choose greengate_bundle.zip from your computer ...')
        up = files.upload()
        name = next(iter(up))
        with open(ZIP, 'wb') as fh:
            fh.write(up[name])
subprocess.run(['rm', '-rf', '/content/gg'], check=True)
subprocess.run(['unzip', '-q', '-o', ZIP, '-d', '/content/gg'], check=True)
print('Bundle ready:', sorted(os.listdir('/content/gg')))

In [ ]:
%%bash
set -e
cd /content
if [ ! -x /content/blender/blender ]; then
  for m in https://mirrors.ocf.berkeley.edu/blender/release https://ftp.nluug.nl/pub/graphics/blender/release \
           https://mirror.clarkson.edu/blender/release https://mirrors.dotsrc.org/blender/blender-release \
           https://download.blender.org/release; do
    echo "Downloading Blender 4.5.14 from $m"
    if wget -q --tries=2 --timeout=90 "$m/Blender4.5/blender-4.5.14-linux-x64.tar.xz" -O blender.tar.xz \
       && [ $(stat -c%s blender.tar.xz) -gt 100000000 ]; then break; fi
  done
  tar -xf blender.tar.xz && mv blender-4.5.14-linux-x64 blender && rm blender.tar.xz
fi
apt-get -qq update >/dev/null 2>&1 || true
apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 >/dev/null 2>&1 || true
/content/blender/blender -b --version | head -1

In [ ]:
# ---- Render (resumable) ---------------------------------------------------
import glob, time, re
FR = os.path.join(WORK, 'frames_' + QUALITY)
os.makedirs(FR, exist_ok=True)
for f in glob.glob(FR + '/f_*.jpg'):          # remove placeholders left by an interrupted run
    if os.path.getsize(f) < 1000:
        os.remove(f)
done0 = len(glob.glob(FR + '/f_*.jpg'))
res, spp = {'final': ('1920x1080', '48'), 'standard': ('1920x1080', '24'), 'fast': ('1600x900', '16'),
            'draft': ('960x540', '8')}[QUALITY]
print(f'{done0}/1440 frames already rendered - continuing ({QUALITY}: {res}, {spp} samples)')
cmd = ['/content/blender/blender', '-b', '--factory-startup', '--python', '/content/gg/build_commercial.py', '--',
       '--res', res, '--samples', spp, '--save', os.path.join(WORK, 'greengate_commercial.blend'),
       '--render', '--out', FR + '/f_####']
t0, n, last = time.time(), 0, ''
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    if 'Saved:' in line and 'f_' in line:
        n += 1
        if n % 10 == 0 or n <= 3:
            el = time.time() - t0
            left = (1440 - done0 - n) * el / n
            print(f'frame {done0 + n}/1440  ({el / n:.1f} s/frame, about {left / 3600:.1f} h left)', flush=True)
    elif any(k in line for k in ('Error', 'Traceback', 'RENDER_DEVICE', 'MISSING_ASSET')):
        print(line.rstrip(), flush=True)
    last = line
p.wait()
total = len([f for f in glob.glob(FR + '/f_*.jpg') if os.path.getsize(f) >= 1000])
print('Rendered frames:', total, '/ 1440')
assert total >= 1440, 'Render stopped early - press Run all again to continue.'

In [ ]:
# ---- Soundtrack + final edit ---------------------------------------------
subprocess.run(['python', '/content/gg/music.py', '/content/gg/music.wav'], check=True)
OUT = os.path.join(WORK, 'GreenGate_Commercial.mp4')
subprocess.run(['python', '/content/gg/finish.py', FR, OUT, '/content/gg/music.wav'], check=True)
if MAKE_SILENT_VERSION:
    subprocess.run(['python', '/content/gg/finish.py', FR, os.path.join(WORK, 'GreenGate_Commercial_no_music.mp4')],
                   check=True)
print('Saved to Drive:', OUT)

In [ ]:
from google.colab import files
files.download(OUT)